# Named Entity Recognition (NER) Pipeline
# Code đầy đủ và data tại: https://github.com/duyhoang17930/Named-Entity-Recognition
## Complete Pipeline: Crawl → Preprocess → label → Encode →  Train → Evaluate → Predict

This notebook contains all steps for building an NER system:
1. **Day1 - Crawl**: Data collection from Google News
2. **Day2 - Preprocess**: Text cleaning and preprocessing
3. **Day3 - Encode**: NER labeling with spaCy
5. **Day4 - Relabeling**: Apply custom entity mappings for improved accuracy
6. **Day5 - Train**: Model training with DistilBERT
7. **Day6 - Results**: Evaluation, visualization, and prediction

# Step 7: Prediction (Inference)

In [41]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Load model
model_path = "./ner_model_final"
print(f"Loading model from {model_path}...")

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)
model.eval()

print("Model loaded!")

Loading model from ./ner_model_final...


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 1939.63it/s]

Model loaded!


In [42]:
# Fix: Load correct id2label from model config instead of label_mappings.json
import json

# Load from model's config (this has the correct mappings)
model_config = json.load(open('./ner_model_final/config.json'))
id2label = model_config['id2label']
label2id = model_config['label2id']

print("Loaded id2label from model config:")
print(id2label)
print(f"\nNumber of labels: {len(id2label)}")

Loaded id2label from model config:
{'0': 'B-CARDINAL', '1': 'B-DATE', '2': 'B-FAC', '3': 'B-GPE', '4': 'B-LANGUAGE', '5': 'B-LOC', '6': 'B-MONEY', '7': 'B-NORP', '8': 'B-ORDINAL', '9': 'B-ORG', '10': 'B-PERSON', '11': 'B-PRODUCT', '12': 'B-QUANTITY', '13': 'B-TIME', '14': 'B-WORK_OF_ART', '15': 'I-CARDINAL', '16': 'I-DATE', '17': 'I-FAC', '18': 'I-GPE', '19': 'I-LOC', '20': 'I-MONEY', '21': 'I-NORP', '22': 'I-ORG', '23': 'I-PERSON', '24': 'I-PRODUCT', '25': 'I-QUANTITY', '26': 'I-TIME', '27': 'I-WORK_OF_ART', '28': 'O'}

Number of labels: 29


In [43]:
def predict_ner(text, show_details=True):
    """
    Predict NER tags for input text.
    
    Args:
        text: Input sentence (string)
        show_details: If True, prints detailed results
    
    Returns:
        List of (word, entity) tuples
    """
    # Tokenize
    tokens = text.split()
    inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True, padding=True)

    # Get predictions
    with torch.no_grad():
        outputs = model(**{k: v.to(model.device) for k, v in inputs.items()})
        predictions = torch.argmax(outputs.logits, dim=2).squeeze().tolist()

    # Map predictions back to words
    word_ids = inputs.word_ids()
    predictions_per_word = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        elif word_idx != previous_word_idx:
            predictions_per_word.append(id2label[str(predictions[idx])])
        previous_word_idx = word_idx
    
    results = list(zip(tokens, predictions_per_word))

    if show_details:
        print(f"\nInput: {text}")
        print("-" * 40)
        print(f"{'TOKEN':<20} {'ENTITY':<15}")
        print("-" * 40)
        for token, pred in results:
            print(f"{token:<20} {pred:<15}")

        # Group entities
        print("\n" + "-" * 40)
        print("Grouped Entities:")
        print("-" * 40)
        current_entity = None
        entity_text = []
        for token, pred in results:
            if pred.startswith('B-'):
                if entity_text:
                    print(f"  {current_entity}: {' '.join(entity_text)}")
                current_entity = pred[2:]
                entity_text = [token]
            elif pred.startswith('I-') and current_entity == pred[2:]:
                entity_text.append(token)
            else:
                if entity_text:
                    print(f"  {current_entity}: {' '.join(entity_text)}")
                    entity_text = []
                current_entity = None
                if not pred.startswith('O'):
                    current_entity = pred[2:]
                    entity_text = [token]
        if entity_text:
            print(f"  {current_entity}: {' '.join(entity_text)}")

    return results

In [48]:
# Test predictions
test_sentences = [
    "Bill Clinton met with Donald Trump in Washington on January 15, 2024.",
    "The United States and China signed a trade agreement in New York.",
    "Apple Inc. CEO Tim Cook announced new products at the event.",
    "Google's headquarters is located in Mountain View, California.",
]

for sentence in test_sentences:
    predict_ner(sentence)
    print()


Input: Bill Clinton met with Donald Trump in Washington on January 15, 2024.
----------------------------------------
TOKEN                ENTITY         
----------------------------------------
Bill                 B-PERSON       
Clinton              I-PERSON       
met                  O              
with                 O              
Donald               B-PERSON       
Trump                I-PERSON       
in                   O              
Washington           B-GPE          
on                   O              
January              B-DATE         
15,                  B-CARDINAL     
2024.                B-DATE         

----------------------------------------
Grouped Entities:
----------------------------------------
  PERSON: Bill Clinton
  PERSON: Donald Trump
  GPE: Washington
  DATE: January
  CARDINAL: 15,
  DATE: 2024.


Input: The United States and China signed a trade agreement in New York.
----------------------------------------
TOKEN                ENTITY     

---

# Summary & Conclusion

## Pipeline Summary

| Step | Description | Output |
|------|-------------|--------|
| Day1 | Data Crawling | 3,680 sentences |
| Day2 | Preprocessing | 3,113 clean sentences |
| Day3 | NER Encoding | ner_dataset.pkl |
| Day4 | Auto-Labeling (Transformers) | manual_labeled.csv |
| Day4.5 | Relabeling (Custom Mappings) | relabeled_output.csv |
| Day5 | Model Training | ner_model_final/ |
| Day6 | Results & Viz | label_distribution.png |

## Model Performance
- Model: DistilBERT (distilbert-base-uncased)
- Training Epochs: 5
- Batch Size: 16
- Learning Rate: 2e-5

## Entities Recognized
- PERSON: People names
- ORG: Organizations
- GPE: Geopolitical entities (countries, cities)
- DATE: Dates
- MONEY: Monetary values
- CARDINAL: Numbers
- NORP: Nationalities/Religious/Political groups
- And more...

## Future Improvements
- Fine-tune with more epochs
- Increase dataset size
- Use larger models (BERT, RoBERTa)
- For Vietnamese: Use PhoBERT
- Implement active learning for better labeling